# Checkpoint 07 — YOLO26n Project8 v0.4, Kaggle 08/10/2026, best epoch 19
Six new sequential method experiments, max 10 epochs each, patience=3.
Initializer SHA256: d6fa68a79abd7eebfc91724c683d2cfd0c2ace6f6623a019fbef44729bf47adb.
07 is the Streamlit dropdown index, not campaign R07 v0.5.
User authorized unreviewed existing v0.5 data on 12/10/2026.
No classroom-domain or independent-test claims. No deployment promotion.


In [ ]:
# 1. Six method experiments; user authorized launch
START_TRAINING = True
ALLOW_UNREVIEWED_PILOT = True  # user authorized exploratory pilots; full review remains incomplete
ARCHIVE_INPUT = ''
DATASET_INPUT = ''
CHECKPOINT_INPUT = ''  # search attached notebook output by SHA256
GPU_IDS = '0,1'
EPOCHS = 10
PATIENCE = 3
IMGSZ = 1280
BATCH = 32
CLOSE_MOSAIC = 2
SMOKE_ONLY = False
RUN_ONE_EPOCH_SMOKE = False
OUTPUT_ROOT = '/kaggle/working/checkpoint07-v04-method-matrix-20261012'
OLD_NAMES = ['person', 'table', 'chair', 'laptop', 'cell phone', 'backpack', 'book', 'cup']
NAMES = OLD_NAMES[:-1] + ['bottle']
EXPECTED_IMAGES = {'train': 12000, 'val': 2500, 'test': 500}
EXPECTED_BOXES = {'train': 33272, 'val': 8359, 'test': 1645}
BEST_SHA256 = 'd6fa68a79abd7eebfc91724c683d2cfd0c2ace6f6623a019fbef44729bf47adb'
MANIFEST_SHA256 = '22a0140ab5e4d454d7e0c46ab24145c6d52d8b949db67ab8142fdd3bb8bb9ccd'
CHECKSUMS_SHA256 = '53a664318ad67bb64f6c894c21e94a6543382b5b9cf859e5ca9d4e283db4a77e'
PORTABLE_YAML_SHA256 = '4b26114dfaf683ce88b3b159cdec3f37f30f9b8168fd260c6cc1a977d718a122'


In [ ]:
# 2. Giữ phiên bản Ultralytics đã dùng tạo checkpoint; không cài requirements CPU local
import subprocess, sys, torch
before_torch = (torch.__version__, torch.version.cuda)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.165'], check=True)
import ultralytics, yaml
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset
from pathlib import Path
import csv, json, hashlib, math, shutil, datetime, gc, zipfile
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
assert ultralytics.__version__ == '8.4.165'
assert before_torch == (torch.__version__, torch.version.cuda), 'Torch/CUDA bị thay đổi; cần reset môi trường.'
print('Ultralytics:', ultralytics.__version__, '| Torch/CUDA:', before_torch)
print('GPU:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# 3. Dataset archive giu nguyen; checkpoint train la checkpoint 07 Project8 v0.4 best epoch 19 theo SHA256
INPUT_ROOT = Path('/kaggle/input')
def file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def safe_path(base, relative):
    path = Path(relative)
    if path.is_absolute() or '..' in path.parts:
        raise ValueError(f'Đường dẫn không an toàn: {relative}')
    result = (base / path).resolve()
    if not result.is_relative_to(base.resolve()):
        raise ValueError(f'Đường dẫn ra ngoài dataset: {relative}')
    return result

# Một file Input; giữ đuôi .zip.bin để Kaggle không tự bung lúc upload.
import stat, time
ARCHIVE_SHA256 = '236972ca3e6142d977a29068cf217dc5312c8fb870c8094a98c4960386a0eb5b'
ARCHIVE_NAME = 'project8_v05_trainval_best_v04_20261008.zip.bin'
ARCHIVE_ROOT = None
archive_paths = [Path(ARCHIVE_INPUT)] if ARCHIVE_INPUT else sorted(INPUT_ROOT.rglob(ARCHIVE_NAME))
if len(archive_paths) > 1:
    raise RuntimeError('Có nhiều archive; đặt ARCHIVE_INPUT cho một file cụ thể.')
if archive_paths:
    archive_path = archive_paths[0]
    print('ARCHIVE_HASH_START:', archive_path, flush=True)
    started = time.perf_counter()
    assert file_sha(archive_path) == ARCHIVE_SHA256, 'Sai SHA256 archive train/val + BEST v0.4'
    ARCHIVE_ROOT = Path('/tmp/project8_v05_15k_runtime') / ('archive_' + ARCHIVE_SHA256[:16])
    ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        infos = archive.infolist()
        member_names = [info.filename for info in infos]
        assert len(member_names) == len(set(member_names)) == 29005, 'Archive thiếu/trùng file'
        manifest_bytes = archive.read('project8_v05/manifest.csv')
        assert hashlib.sha256(manifest_bytes).hexdigest() == MANIFEST_SHA256
        archive_rows = list(csv.DictReader(manifest_bytes.decode('utf-8-sig').splitlines()))
        assert len(archive_rows) == 15000
        expected_members = {
            'project8_v05/' + row[key]
            for row in archive_rows if row['split'] in {'train', 'val'}
            for key in ('image_path', 'label_path')
        } | {
            'project8_v05/manifest.csv', 'project8_v05/checksums.json',
            'project8_v05/data.yaml', 'project8_v05/training_readiness.json',
            'BEST_Project8_v04_20261008.pt',
        }
        assert set(member_names) == expected_members, 'Archive có file dư/thiếu hoặc test media'
        assert shutil.disk_usage(ARCHIVE_ROOT).free > sum(info.file_size for info in infos) + 512*1024**2, 'Thiếu ổ tạm để giải nén'
        # Kiểm toàn bộ đường dẫn trước khi ghi bất kỳ file nào.
        for info in infos:
            assert '\\' not in info.filename and not info.is_dir()
            assert not stat.S_ISLNK(info.external_attr >> 16), 'Không nhận symlink'
            safe_path(ARCHIVE_ROOT, info.filename)
        print('ARCHIVE_HASH_PASS: 29005 files; extract locally', flush=True)
        for index, info in enumerate(infos, 1):
            destination = safe_path(ARCHIVE_ROOT, info.filename)
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as read, destination.open('wb') as write:
                shutil.copyfileobj(read, write, length=1024*1024)
            assert destination.stat().st_size == info.file_size
            if index % 2000 == 0 or index == len(infos):
                print('UNZIP:', index, '/', len(infos), '| seconds:', round(time.perf_counter()-started, 1), flush=True)
    INPUT_ROOT = ARCHIVE_ROOT
    print('ARCHIVE_READY:', INPUT_ROOT, '| seconds:', round(time.perf_counter()-started, 1), flush=True)
else:
    print('Không có archive; dùng Input đã bung sẵn.', flush=True)

manifests = [Path(DATASET_INPUT) / 'manifest.csv'] if DATASET_INPUT else sorted(INPUT_ROOT.rglob('manifest.csv'))
valid_roots = [p.parent for p in manifests if p.is_file() and file_sha(p) == MANIFEST_SHA256 and (p.parent/'images/train').is_dir()]
if not valid_roots:
    raise FileNotFoundError('Chưa gắn đúng dataset v0.5 15k. Add Input dataset một file .zip.bin hoặc dataset v0.5 đã bung sẵn.')
if len(valid_roots) != 1:
    raise RuntimeError('Có nhiều bản dataset 15k; đặt DATASET_INPUT cho một bản cụ thể.')
SOURCE_DATA = valid_roots[0]
assert file_sha(SOURCE_DATA / 'checksums.json') == CHECKSUMS_SHA256
checksums = json.loads((SOURCE_DATA / 'checksums.json').read_text(encoding='utf-8'))
with (SOURCE_DATA / 'manifest.csv').open(encoding='utf-8-sig', newline='') as stream:
    rows = list(csv.DictReader(stream))
assert len(rows) == 15000 and Counter(r['split'] for r in rows) == EXPECTED_IMAGES
cfg = yaml.safe_load((SOURCE_DATA/'data.yaml').read_text(encoding='utf-8'))
names = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][i] for i in range(8)]
assert names == NAMES and cfg['nc'] == 8
assert len({r['pixel_sha256'] for r in rows}) == 15000
expected_files = {r[k] for r in rows for k in ('image_path','label_path')}
assert set(checksums) == expected_files | {'manifest.csv','data.yaml'} and len(checksums) == 30002
assert checksums['manifest.csv'] == MANIFEST_SHA256
# Export YAML is portable; source data.yaml digest remains in the historical release checksums.
assert file_sha(SOURCE_DATA/'data.yaml') == PORTABLE_YAML_SHA256
assert cfg.get('train')=='images/train' and cfg.get('val')=='images/val' and not cfg.get('test')

weight_paths = [Path(CHECKPOINT_INPUT)] if CHECKPOINT_INPUT else sorted(set(Path('/kaggle/input').rglob('*.pt')) | set(INPUT_ROOT.rglob('*.pt')))
matched_weights = [p for p in weight_paths if p.is_file() and file_sha(p) == BEST_SHA256]
if not matched_weights:
    raise FileNotFoundError('Thieu checkpoint 07 Project8 v0.4 best epoch 19 dung SHA256; Add Input output notebook v0.5.')
initial = matched_weights[0]  # các bản còn lại, nếu có, đều cùng byte/hash
model_check = YOLO(str(initial))
assert model_check.task == 'detect' and [model_check.names[i] for i in range(8)] == NAMES
print('Dataset:', SOURCE_DATA, '| split:', dict(Counter(r['split'] for r in rows)))
print('checkpoint 07 Project8 v0.4 best epoch 19:', initial, '| SHA256:', file_sha(initial))
print('ID 7 bottle gồm cup+bottle; giữ nguyên 8 ID từ best v0.4.')
del model_check; gc.collect()
readiness = json.loads((SOURCE_DATA/'training_readiness.json').read_text(encoding='utf-8'))
assert readiness['manifest_sha256'] == MANIFEST_SHA256
TRAIN_GATE = (readiness.get('training_ready') is True and readiness.get('full_visual_review_complete') is True and all(r['review_stage']=='full_review_complete' for r in rows))
print('Full visual review gate:', TRAIN_GATE)

assert {split:sum(int(r['boxes']) for r in rows if r['split']==split) for split in EXPECTED_IMAGES} == EXPECTED_BOXES


In [ ]:
# 4. Stage và kiểm SHA256/tọa độ toàn bộ train+val; test không vào thư mục train
WORK = Path('/tmp/project8_v05_15k_runtime')
DATA_DIR = SOURCE_DATA if ARCHIVE_ROOT is not None else WORK / 'dataset'
DATA_DIR.mkdir(parents=True, exist_ok=True)
train_val = [r for r in rows if r['split'] in {'train','val'}]

def stage_record(row):
    for key, hash_key in [('image_path','image_sha256'),('label_path','label_sha256')]:
        src = safe_path(SOURCE_DATA, row[key]); dst = safe_path(DATA_DIR, row[key])
        assert checksums[row[key]] == row[hash_key]
        assert file_sha(src) == row[hash_key], f'File nguồn bị đổi: {src}'
        if src != dst:
            dst.parent.mkdir(parents=True, exist_ok=True)
            if not dst.is_file() or file_sha(dst) != row[hash_key]:
                # Input đã bung: đọc/ghi thường để tránh sendfile trên mount.
                with src.open('rb') as read, dst.open('wb') as write:
                    shutil.copyfileobj(read, write, length=1024*1024)
            assert file_sha(dst) == row[hash_key], dst
        # Archive: file đã trên ổ tạm và vừa được hash; không chép/hash lần hai.
    counts = Counter()
    for line in (DATA_DIR/row['label_path']).read_text(encoding='utf-8').splitlines():
        values = list(map(float, line.split()))
        assert len(values)==5 and all(math.isfinite(v) for v in values)
        cls,x,y,w,h = values
        assert cls.is_integer() and 0<=cls<8 and 0<=x<=1 and 0<=y<=1 and 0<w<=1 and 0<h<=1
        counts[int(cls)] += 1
    assert sum(counts.values()) == int(row['boxes']) and counts
    return counts

class_counts = Counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    for i, counts in enumerate(pool.map(stage_record, train_val), 1):
        class_counts.update(counts)
        if i % 2000 == 0 or i == len(train_val):
            print('Stage/hash/labels:', i, '/', len(train_val), flush=True)
assert sum(class_counts.values()) == 41631
assert not (DATA_DIR/'images/test').exists() and not (DATA_DIR/'labels/test').exists()
actual_files = {p.relative_to(DATA_DIR).as_posix() for folder in ['images','labels'] for p in (DATA_DIR/folder).rglob('*') if p.is_file() and not p.name.endswith('.cache')}
assert actual_files == {r[k] for r in train_val for k in ('image_path','label_path')}, 'Có file dư trong scratch; dùng session mới.'
DATA_YAML = WORK/'training_data.yaml'
DATA_YAML.write_text(yaml.safe_dump({'path':str(DATA_DIR.resolve()), 'train':'images/train', 'val':'images/val', 'nc':8, 'names':dict(enumerate(NAMES))},sort_keys=False),encoding='utf-8')
resolved = check_det_dataset(str(DATA_YAML),autodownload=False)
assert not resolved.get('test')
for split in ['train','val']:
    assert Path(resolved[split]) == (DATA_DIR/'images'/split).resolve()
OUT = Path(OUTPUT_ROOT); OUT.mkdir(parents=True, exist_ok=True)
PREPARED = {'dataset_manifest_sha256':MANIFEST_SHA256,'checkpoint_sha256':BEST_SHA256,'all_input_images':EXPECTED_IMAGES,'training_images':len(train_val),'training_boxes':41631,'classes':NAMES,'class_id_7':'cups+bottles by user request','resume':False,'cls_remap':False}
(OUT/'prepared.json').write_text(json.dumps(PREPARED,indent=2),encoding='utf-8')
print('PREPARED: 12.000 train + 2.500 val; 500 test giữ riêng. Chưa train.')
print('Box train+val:', {NAMES[i]:class_counts[i] for i in range(8)})

In [ ]:
"""Six controlled trials from checkpoint 07, Project8 v0.4 best epoch 19."""


def build_matrix(initial, data, output, backbone_layers):
    if not backbone_layers or any(type(i) is not int or i < 0 for i in backbone_layers):
        raise ValueError('Supply verified backbone layer indices')
    common = dict(model=str(initial), data=str(data), project=str(output),
                  epochs=10, patience=3, resume=False, exist_ok=False,
                  seed=42, deterministic=True, pretrained=True, cls_remap=False,
                  imgsz=1280, batch=32, device='0,1', workers=2, cache=False,
                  amp=True, val=True, plots=True, save=True, save_period=1,
                  optimizer='MuSGD', lr0=.0001, lrf=.01, cos_lr=False,
                  warmup_epochs=1., warmup_bias_lr=.0001, warmup_momentum=.8,
                  momentum=.937, weight_decay=.0005, nbs=64,
                  hsv_h=.015, hsv_s=.7, hsv_v=.4, degrees=0., translate=.1,
                  scale=.5, shear=0., perspective=0., flipud=0., fliplr=.5,
                  mosaic=.5, close_mosaic=2, mixup=0., cutmix=0., copy_paste=0.,
                  box=7.5, cls=.5, dfl=1.5, cls_pw=0., multi_scale=0.,
                  rect=False, fraction=1., compile=False, freeze=None)
    changes = [dict(name='E1_MuSGD_lr1e4'), dict(name='E2_MuSGD_lr3e4', lr0=.0003),
               dict(name='E3_AdamW_lr1e4', optimizer='AdamW'),
               dict(name='E4_AdamW_lr3e4', optimizer='AdamW', lr0=.0003),
               dict(name='E5_MuSGD_scale02', scale=.2),
               dict(name='E6_MuSGD_frozen_backbone', freeze=list(backbone_layers))]
    return [dict(common, **change) for change in changes]

"""Sequential exploratory training; execute after verified staging cells."""
import inspect
import time
import traceback
import os
import signal
import subprocess
from ultralytics.utils.torch_utils import EarlyStopping

assert START_TRAINING and ALLOW_UNREVIEWED_PILOT
assert EPOCHS == 10 and PATIENCE == 3
assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 2
assert file_sha(initial) == BEST_SHA256
assert ultralytics.__version__ == '8.4.165'

CHECKPOINT_NAME = 'checkpoint 07 — YOLO26n Project8 v0.4, phiên Kaggle ngày 08/10/2026, best epoch 19'
print(CHECKPOINT_NAME, initial, BEST_SHA256, flush=True)
print('EXPLORATORY: unreviewed v0.5 labels; no classroom improvement claim; no test selection.', flush=True)

probe = YOLO(str(initial))
assert probe.model.yaml.get('scale') == 'n'
backbone_count = len(probe.model.yaml['backbone'])
assert 0 < backbone_count < len(probe.model.model)
backbone_layers = list(range(backbone_count))
print('Verified backbone:', [(i, type(probe.model.model[i]).__name__) for i in backbone_layers], flush=True)
del probe; gc.collect(); torch.cuda.empty_cache()
jobs = build_matrix(initial, DATA_YAML, OUT/'train', backbone_layers)

# Pin observed stopper semantics; strict positive-fitness ties must not reset patience.
stopper_source = inspect.getsource(EarlyStopping)
stopper = EarlyStopping(patience=3)
assert [stopper(i+1, score) for i, score in enumerate([.5, .5, .5, .5])] == [False, False, False, True]
zero_stopper = EarlyStopping(patience=3)
zero_behavior = [zero_stopper(i+1, 0.) for i in range(4)]
(OUT/'early_stopping_source.py').write_text(stopper_source, encoding='utf-8')

started = time.monotonic()
state = dict(status='preflight_pass', checkpoint_name=CHECKPOINT_NAME,
             initial_sha256=BEST_SHA256, manifest_sha256=MANIFEST_SHA256,
             full_visual_review_complete=bool(TRAIN_GATE),
             user_authorized_unreviewed_method_experiment=True,
             classroom_claim_allowed=False, test_used=False, jobs=jobs, runs={},
             environment=dict(ultralytics=ultralytics.__version__, torch=torch.__version__,
                              cuda=torch.version.cuda, gpus=[torch.cuda.get_device_name(i) for i in (0, 1)]),
             early_stopper_zero_behavior=zero_behavior,
             evaluation_note='Native trainer AP1280 first; COCO640 and application evaluation are separate follow-up artifacts.')


def preserve_state():
    state['elapsed_seconds'] = round(time.monotonic()-started, 1)
    temporary = OUT/'status.json.partial'
    temporary.write_text(json.dumps(state, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    temporary.replace(OUT/'status.json')


def archive_trial(label):
    trial = OUT/'train'/label
    if not trial.exists():
        return
    bundle = Path(shutil.make_archive(str(OUT/label), 'zip', root_dir=trial))
    with zipfile.ZipFile(bundle) as archive:
        assert archive.testzip() is None
    state['runs'][label]['archive'] = dict(path=str(bundle), sha256=file_sha(bundle), bytes=bundle.stat().st_size)
    preserve_state()
    print('ARCHIVE_VERIFIED', label, file_sha(bundle), flush=True)


def native_eval(weights, label):
    detector = YOLO(str(weights))
    metrics = detector.val(data=str(DATA_YAML), split='val', imgsz=1280,
                           batch=8, device=0, workers=2, conf=.001, iou=.7,
                           nms=False, max_det=300, augment=False, plots=True,
                           project=str(OUT/'evaluation'), name=label, exist_ok=False)
    result = dict(mAP50=float(metrics.box.map50), mAP50_95=float(metrics.box.map),
                  per_class_AP50_95=dict(zip(NAMES, map(float, metrics.box.maps))),
                  protocol=dict(imgsz=1280, conf=.001, iou=.7, nms=False, split='val'),
                  checkpoint_sha256=file_sha(weights))
    del detector; gc.collect(); torch.cuda.empty_cache()
    return result


def train_ddp(config, record):
    """CLI spawns DDP; parent reads persisted epoch history, not child callbacks."""
    config_path = OUT/(config['name']+'_config.yaml')
    config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
    log_path = OUT/(config['name']+'_training.log')
    history_path = Path(config['project'])/config['name']/'results.csv'
    launched = time.monotonic()
    record['log'] = str(log_path)
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen([shutil.which('yolo'), 'detect', 'train', 'cfg='+str(config_path)],
                                   stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        previous_epoch = 0
        while process.poll() is None:
            if history_path.is_file():
                with history_path.open(newline='') as stream:
                    history = [{k.strip(): v.strip() for k, v in row.items()} for row in csv.DictReader(stream)]
                # Ignore a last row still being written; trainer writes metrics before weights.
                if history and history[-1].get('metrics/mAP50-95(B)'):
                    latest = history[-1]
                    epoch = int(latest['epoch'])
                    fitness = float(latest['metrics/mAP50-95(B)'])
                    if not math.isfinite(fitness) or epoch > config['epochs']:
                        os.killpg(process.pid, signal.SIGTERM)
                        process.wait()
                        raise ValueError('Invalid fitness or epoch cap exceeded')
                    if epoch != previous_epoch:
                        record.update(status='running', epochs_completed=epoch, fitness=fitness,
                                      epoch_metrics=history, seconds=time.monotonic()-launched)
                        previous_epoch = epoch
                        preserve_state()
                        print('PROGRESS', config['name'], 'epoch', epoch, '/', config['epochs'],
                              'mAP50-95', fitness, flush=True)
            if time.monotonic()-launched >= 110*60 or time.monotonic()-started >= 9.5*3600:
                record['budget_stop'] = True
                os.killpg(process.pid, signal.SIGTERM)
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL)
                    process.wait()
                record['status'] = 'budget_stop'
                preserve_state()
                return
            time.sleep(10)
        if process.returncode != 0:
            print(log_path.read_text(encoding='utf-8')[-8000:], flush=True)
            raise RuntimeError('DDP training failed; exit '+str(process.returncode))
    with history_path.open(newline='') as stream:
        history = [{k.strip(): v.strip() for k, v in row.items()} for row in csv.DictReader(stream)]
    record.update(epochs_completed=len(history), epoch_metrics=history, seconds=time.monotonic()-launched)
    assert 0 < len(history) <= config['epochs']
    print(log_path.read_text(encoding='utf-8')[-3500:], flush=True)


preserve_state()
state['status'] = 'baseline_validation'
preserve_state()
state['baseline_checkpoint07_native'] = native_eval(initial, 'checkpoint07_raw_native')
preserve_state()

# Smoke is separate and never becomes an initializer for E1–E6.
state['status'] = 'smoke_running'; preserve_state()
smoke_config = dict(jobs[0], name='SMOKE_checkpoint07', epochs=1, patience=3)
smoke_started = time.monotonic()
state['smoke'] = dict(status='starting', epochs_completed=0)
train_ddp(smoke_config, state['smoke'])
assert not state['smoke'].get('budget_stop')
state['smoke_seconds'] = time.monotonic()-smoke_started
state['estimated_training_seconds_max'] = state['smoke_seconds']*60
gc.collect(); torch.cuda.empty_cache()
preserve_state()
if state['estimated_training_seconds_max'] > 9.5*3600:
    state['status'] = 'budget_review_required'; preserve_state()
    raise RuntimeError('Measured estimate exceeds ten-hour budget with archive reserve; no matrix jobs launched.')

for config in jobs:
    label = config['name']
    if time.monotonic()-started >= 9.5*3600:
        state['status'] = 'budget_stop_before_next_job'; preserve_state()
        break
    assert file_sha(initial) == BEST_SHA256
    state['status'] = 'training'
    state['current_run'] = label
    state['runs'][label] = dict(status='starting', config=config,
                              initial_sha256=file_sha(initial), epochs_completed=0)
    preserve_state()
    run_started = time.monotonic()
    try:
        record = state['runs'][label]
        train_ddp(config, record)
        best = Path(config['project'])/label/'weights/best.pt'
        assert best.is_file() and record['epochs_completed'] <= 10
        record.update(status='budget_stop' if record.get('budget_stop') else
                      ('early_stop' if record['epochs_completed'] < 10 else 'completed'),
                      checkpoint=str(best), checkpoint_sha256=file_sha(best),
                      seconds=time.monotonic()-run_started)
        archive_trial(label)
        gc.collect(); torch.cuda.empty_cache()
        record['native_validation'] = native_eval(best, label+'_native')
        preserve_state()
        if record.get('budget_stop'):
            state['status'] = 'budget_stop'; preserve_state(); break
    except Exception:
        state['runs'][label].update(status='failed', error=traceback.format_exc())
        state['status'] = 'failed_requires_review'
        preserve_state(); archive_trial(label)
        raise
else:
    state['status'] = 'training_complete_evaluation_followup_required'
    preserve_state()
    print('MATRIX_TRAINING_COMPLETE: native val available; no classroom or independent-test claim.', flush=True)
